# Imports

In [2]:
import os
import pickle
import re
import shutil
import sys
sys.path.append(os.path.dirname(os.getcwd()))
from itertools import product
from scipy.stats import norm
import matplotlib
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
import yaml
from matplotlib.backends.backend_pdf import PdfPages
from tools import load_npy, load_yaml_as_df, load_pkl, exist_metric, exist_stf_metric, inverse_stf_metrics, keep_split, is_full_group, load_metric_from_log

plt.style.use('default')
matplotlib.rcParams['pdf.fonttype'] = 42
matplotlib.rcParams['ps.fonttype'] = 42
plt.rc('font', family='Arial')
matplotlib.rcParams['mathtext.fontset'] = 'stix'
matplotlib.rcParams['font.size'] = 10

pd.set_option('display.max_columns', None)
pd.set_option('display.max_rows', None)

# Varying input length

## load data

In [3]:
root = '/mnt/tidalfs-bdsz01/usr/panlicheng/workspace/ProjDF-Meta/results_ML3/vary_input'
exp_dirs = os.listdir(root)
exp_dirs = [os.path.join(root, exp_dir) for exp_dir in exp_dirs]

params = ['model', 'seq_len', 'pred_len', 'data_id', 'learning_rate', 'inner_lr', 'meta_lr', 'rec_lambda', 'auxi_lambda', 'reg_lambda', 'lradj', 'train_epochs', 'patience', 'batch_size', 'auxi_batch_size', 'warmup_steps', 'meta_inner_steps', 'overlap_ratio', 'num_tasks', 'max_norm', 'auxi_loss', 'first_order', 'dropout', 'cycle', 'task_name']
metric_names = ['mse', 'mae', 'cov']

df = []
for exp_dir in exp_dirs:
    runned, setting_dir = exist_metric(exp_dir)
    if not runned:
        continue

    config = load_yaml_as_df(os.path.join(setting_dir, 'config.yaml'))
    metric = load_npy(os.path.join(setting_dir, 'metrics.npy'))
    result = config[params]
    # if 'PatchTST' in exp_dir and 'all' in exp_dir and 'ml3' not in result['task_name']:
    #     shutil.rmtree(exp_dir)
    #     continue
    if len(metric) == 6:
        log_metrics = load_metric_from_log(os.path.join(exp_dir, 'result_long_term_forecast.txt'))
        cov_loss = log_metrics['cov'] if log_metrics and 'cov' in log_metrics else np.inf
        result.loc[:, metric_names] = metric[1], metric[0], cov_loss
    else:
        result.loc[:, metric_names] = metric[1], metric[0], metric[2]
    result.loc[:, ['meta_type']] = config[['meta_type']] if 'meta_type' in config.columns else 'all'
    result.loc[:, ['exp_dir']] = exp_dir
    df.append(result)

df = pd.concat(df, ignore_index=True)

df.sort_values(by=['model', 'data_id', 'seq_len', 'pred_len'], inplace=True)

save_root = '/mnt/tidalfs-bdsz01/usr/panlicheng/workspace/ProjDF-Meta/stats_ML3'
os.makedirs(save_root, exist_ok=True)
df.to_csv(f"{save_root}/varying_input.csv", index=False)

df.head(4)

,model,seq_len,pred_len,data_id,learning_rate,inner_lr,meta_lr,rec_lambda,auxi_lambda,reg_lambda,lradj,train_epochs,patience,batch_size,auxi_batch_size,warmup_steps,meta_inner_steps,overlap_ratio,num_tasks,max_norm,auxi_loss,first_order,dropout,cycle,task_name,mse,mae,cov,meta_type,exp_dir
135,PatchTST,96,96,Weather,0.0002,0.0002,0.05,1.0,0.0,0.0,type1,3,3,32,64,300,1,0.0,3,5.0,MSE,1,0.1,24,long_term_forecast_meta_ml3,0.180137,0.223513,0.084849,all,/mnt/tidalfs-bdsz01/usr/panlicheng/workspace/P...
136,PatchTST,96,96,Weather,0.0002,0.0002,0.10,1.0,0.0,0.0,type1,3,3,32,64,300,1,0.0,3,5.0,MSE,1,0.1,24,long_term_forecast_meta_ml3,0.181852,0.226392,0.068509,all,/mnt/tidalfs-bdsz01/usr/panlicheng/workspace/P...
137,PatchTST,96,96,Weather,0.0002,0.0002,0.20,1.0,0.0,0.0,type1,3,3,32,64,300,1,0.0,3,5.0,MSE,1,0.1,24,long_term_forecast_meta_ml3,0.183564,0.228586,0.053923,all,/mnt/tidalfs-bdsz01/usr/panlicheng/workspace/P...
138,PatchTST,96,96,Weather,0.0005,0.0005,0.05,1.0,0.0,0.0,type1,3,3,32,64,300,1,0.0,3,5.0,MSE,1,0.1,24,long_term_forecast_meta_ml3,0.186109,0.229123,0.087061,all,/mnt/tidalfs-bdsz01/usr/panlicheng/workspace/P...


## preprocess

In [4]:
stats_root = '/mnt/tidalfs-bdsz01/usr/panlicheng/workspace/ProjDF-Meta/stats_ML3'
log_root = '/mnt/tidalfs-bdsz01/usr/panlicheng/workspace/ProjDF-Meta/logs'
baselines = pd.read_csv(f'{log_root}/baselines_chosen.csv')
finetunes_best = pd.read_csv(f'{stats_root}/finetune_best.csv')

base = baselines.copy()
base = base[
    ((base.data_id == 'Weather') & (base.model.isin(['TQNet', 'PatchTST'])))
]
base['seq_len'] = 96
base['label'] = 'DF'

best = finetunes_best.copy()
best = best[
    ((best.data_id == 'Weather') & (best.model.isin(['TQNet', 'PatchTST'])))
]
best['seq_len'] = 96
best['label'] = 'QDF'


df2 = df.copy()
df2_base = df2[df2.task_name.apply(lambda x: 'ml3' not in x)].copy()
df2_base['label'] = 'DF'

df2 = df2[df2.task_name.apply(lambda x: 'ml3' in x)].copy()
df2['label'] = 'QDF'

min_mse_idx = df2.groupby(['model', 'data_id', 'seq_len', 'pred_len'])['mse'].idxmin()
df2 = df2.loc[min_mse_idx]

columns = ['model', 'data_id', 'seq_len', 'pred_len', 'label', 'mse', 'mae']
vary_seq = pd.concat([base[columns], df2_base[columns], best[columns], df2[columns]], ignore_index=True)
vary_seq['seq_len'] = vary_seq['seq_len'].astype(int)
vary_seq['pred_len'] = vary_seq['pred_len'].astype(int)

dst_order = ['Weather']
vary_seq['data_id'] = pd.Categorical(vary_seq['data_id'], categories=dst_order, ordered=True)

model_order = ['TQNet', 'PatchTST']
vary_seq['model'] = pd.Categorical(vary_seq['model'], categories=model_order, ordered=True)

label_order = ['QDF', 'DF']
vary_seq['label'] = pd.Categorical(vary_seq['label'], categories=label_order, ordered=True)

vary_seq_avg = vary_seq.groupby(['model', 'data_id', 'seq_len', 'label']).mean(numeric_only=True).reset_index()
vary_seq_avg['pred_len'] = 'Avg'
vary_seq = pd.concat([vary_seq, vary_seq_avg], ignore_index=True)

vary_seq.sort_values(by=['model', 'data_id', 'label', 'seq_len', 'pred_len'], inplace=True)

# save_root = '/data/home/Licheng/workspace/TSF-PCA/stats_PCA'
# vary_seq.round(3).to_csv(f'{save_root}/vary_seq.csv', index=False, float_format='%.3f')
vary_seq.dropna(inplace=True, thresh=6)
vary_seq

/tmp/ipykernel_1485149/1876595297.py:45: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  vary_seq_avg = vary_seq.groupby(['model', 'data_id', 'seq_len', 'label']).mean(numeric_only=True).reset_index()


,model,data_id,seq_len,pred_len,label,mse,mae
32,TQNet,Weather,96,96,QDF,0.158300,0.200693
33,TQNet,Weather,96,192,QDF,0.206637,0.244936
34,TQNet,Weather,96,336,QDF,0.262811,0.286290
35,TQNet,Weather,96,720,QDF,0.342250,0.339154
64,TQNet,Weather,96,Avg,QDF,0.242499,0.267768
52,TQNet,Weather,192,96,QDF,0.151933,0.199025
53,TQNet,Weather,192,192,QDF,0.197746,0.240933
54,TQNet,Weather,192,336,QDF,0.251799,0.281608
55,TQNet,Weather,192,720,QDF,0.324353,0.332215
66,TQNet,Weather,192,Avg,QDF,0.231458,0.263445


## write to table

In [5]:
contents = []


for sl in [96, 192, 336, 720]:
    line = r"& \multirow{5}{*}{" + str(sl) + r"}"
    contents.append(line)
    _df = vary_seq[vary_seq.seq_len == sl].copy().reset_index(drop=True)
    for pl in [96, 192, 336, 720, 'Avg']:
        line = f"& {pl} " if pl == 96 else f"&& {pl} "
        _dfp = _df[_df.pred_len == pl].copy().reset_index(drop=True)
        for model in ['TQNet', 'PatchTST']:
            _dfm = _dfp[_dfp.model == model].copy().reset_index(drop=True)
            for row in _dfm.itertuples():
                line += f"& {row.mse:.3f} & {row.mae:.3f} "
        line += r"\\"
        contents.append(line)
        if pl == 720:
            contents.append(r"\cmidrule(lr){3-11}")
    if sl != 720:
        contents.append(r"\cmidrule(lr){2-11}" + '\n')

print('\n'.join(contents))

& \multirow{5}{*}{96}
& 96 & 0.158 & 0.201 & 0.160 & 0.203 & 0.180 & 0.224 & 0.189 & 0.230 \\
&& 192 & 0.207 & 0.245 & 0.210 & 0.247 & 0.226 & 0.262 & 0.228 & 0.262 \\
&& 336 & 0.263 & 0.286 & 0.267 & 0.289 & 0.279 & 0.300 & 0.288 & 0.305 \\
&& 720 & 0.342 & 0.339 & 0.346 & 0.342 & 0.354 & 0.347 & 0.362 & 0.354 \\
\cmidrule(lr){3-11}
&& Avg & 0.242 & 0.268 & 0.246 & 0.270 & 0.260 & 0.283 & 0.267 & 0.288 \\
\cmidrule(lr){2-11}

& \multirow{5}{*}{192}
& 96 & 0.152 & 0.199 & 0.151 & 0.197 & 0.161 & 0.208 & 0.163 & 0.209 \\
&& 192 & 0.198 & 0.241 & 0.198 & 0.241 & 0.207 & 0.248 & 0.207 & 0.249 \\
&& 336 & 0.252 & 0.282 & 0.253 & 0.283 & 0.259 & 0.287 & 0.268 & 0.293 \\
&& 720 & 0.324 & 0.332 & 0.327 & 0.334 & 0.334 & 0.337 & 0.511 & 0.451 \\
\cmidrule(lr){3-11}
&& Avg & 0.231 & 0.263 & 0.232 & 0.264 & 0.240 & 0.270 & 0.287 & 0.301 \\
\cmidrule(lr){2-11}

& \multirow{5}{*}{336}
& 96 & 0.148 & 0.198 & 0.149 & 0.198 & 0.160 & 0.214 & 0.158 & 0.208 \\
&& 192 & 0.195 & 0.240 & 0.196 & 0.243 & 0